In [ ]:
#01_main_training_pipeline

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, accuracy_score, roc_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure GPU memory growth
try:
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
        print("Using GPU:", physical_devices[0])
    else:
        print("Using CPU")
except RuntimeError as e:
    print("Warning: Could not set GPU memory growth:", str(e))

# Set random seed
tf.random.set_seed(42)
np.random.seed(42)

# Define paths
data_dir = r"D:\SCI\DAIC-WOZ\code"
csv_files = ["goemotions_1.csv", "goemotions_2.csv", "goemotions_3.csv"]
model_path = os.path.join(data_dir, "goemotions_bert_model")
output_dir = data_dir
os.makedirs(output_dir, exist_ok=True)

# Verify file existence
for file in csv_files:
    file_path = os.path.join(data_dir, file)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}. Download from: https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/{file}")

# Load and concatenate CSV files
dataframes = [pd.read_csv(os.path.join(data_dir, file)) for file in csv_files]
data = pd.concat(dataframes, ignore_index=True)

# Check for duplicates
print("Duplicate IDs before deduplication:", data['id'].duplicated().sum())

# Define emotions
emotion_columns = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
depression_emotions = ['sadness', 'grief', 'remorse', 'disappointment', 'disapproval']
positive_emotions = ['admiration', 'amusement', 'approval', 'caring', 'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief']

# === 12 个中性情绪===
neutral_emotions = [
    'anger', 'annoyance', 'confusion', 'curiosity', 'desire',
    'disgust', 'embarrassment', 'fear', 'nervousness', 'realization', 'surprise', 'neutral'
]

# Aggregate annotations per id
agg_dict = {col: 'sum' for col in emotion_columns}
agg_dict.update({
    'text': 'first', 'author': 'first', 'subreddit': 'first', 'link_id': 'first',
    'parent_id': 'first', 'created_utc': 'first', 'rater_id': 'first', 'example_very_unclear': 'first'
})
data = data.groupby('id').agg(agg_dict).reset_index()

# Convert summed labels to binary
for col in emotion_columns:
    data[col] = (data[col] >= 1).astype(int)

# Filter out unclear examples
data = data[data['example_very_unclear'] == False]

# Rater agreement filter for rare emotions
rare_emotions = ['grief', 'pride', 'relief', 'nervousness']
data['rare_sum'] = data[rare_emotions].sum(axis=1)
data = data[(data[emotion_columns].sum(axis=1) >= 1) | (data['rare_sum'] >= 2)]
data = data.drop(columns=['rare_sum'])



# Aggregate per author
author_data = data.groupby('author').agg({
    **{col: 'sum' for col in emotion_columns},
    'text': lambda x: ' '.join(x),
    'subreddit': 'first',
    'id': 'count'
}).rename(columns={'id': 'comment_count'}).reset_index()

# Filter authors with >=2 comments
author_data = author_data[author_data['comment_count'] >= 2]
print(f"Filtered to authors with >=2 comments: {len(author_data)}")


author_data['DEDI'] = (
    author_data[depression_emotions].sum(axis=1) /
    (author_data[depression_emotions].sum(axis=1) +
     author_data[positive_emotions].sum(axis=1) +
     author_data[neutral_emotions].sum(axis=1))
)

# Create ground-truth depression labels
prevalence = 0.1102
threshold = author_data['DEDI'].quantile(1 - prevalence)
author_data['depression_true'] = (author_data['DEDI'] >= threshold).astype(int)
print(f"Depression threshold (for {prevalence*100:.2f}% prevalence): {threshold:.4f}")
print(f"True Positive Rate (Prevalence): {author_data['depression_true'].mean():.4f}")

# === Data Distribution Analysis ===
print("\n=== Data Distribution Analysis ===")
print(f"Number of unique authors: {len(author_data)}")
print(f"Comment count distribution:")
print(author_data['comment_count'].describe())
plt.figure(figsize=(8, 6))
author_data['comment_count'].hist(bins=20, color='#1f77b4')
plt.title('Distribution of Comments per Author')
plt.xlabel('Number of Comments')
plt.ylabel('Frequency')
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'comment_count_distribution.png'))
plt.close()

print("\nEmotion Prevalence (True Labels, per-author):")
emotion_prevalence = (author_data[emotion_columns] > 0).mean().sort_values(ascending=False)
print(emotion_prevalence)
plt.figure(figsize=(12, 6))
emotion_prevalence.plot(kind='bar', color='#1f77b4')
plt.title('Emotion Prevalence in Authors')
plt.xlabel('Emotion')
plt.ylabel('Proportion of Authors')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'emotion_prevalence_author.png'))
plt.close()

print("\nDEDI Distribution:")
print(author_data['DEDI'].describe())
plt.figure(figsize=(8, 6))
author_data['DEDI'].hist(bins=50, color='blue', range=(0, 1))
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold = {threshold:.4f}')
plt.title('Distribution of Depression Emotion Proportion (DEDI)')
plt.xlabel('DEDI')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'DEDI_distribution_author.png'))
plt.close()

print("\nTop 10 Subreddits by Author Count:")
subreddit_counts = author_data['subreddit'].value_counts().head(10)
print(subreddit_counts)
plt.figure(figsize=(12, 6))
subreddit_counts.plot(kind='bar', color='#1f77b4')
plt.title('Top 10 Subreddits by Author Count')
plt.xlabel('Subreddit')
plt.ylabel('Number of Authors')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'subreddit_distribution_author.png'))
plt.close()

# Split data into train and validation sets
train_data, val_data = train_test_split(author_data, test_size=0.2, random_state=42, stratify=author_data['depression_true'])

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained(model_path)

# Tokenize texts
max_length = 100
def tokenize_texts(texts):
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    return encodings

train_encodings = tokenize_texts(train_data['text'])
val_encodings = tokenize_texts(val_data['text'])

# Extract TF-IDF features
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
tfidf.fit(author_data['text'])
train_tfidf = tfidf.transform(train_data['text']).toarray()
val_tfidf = tfidf.transform(val_data['text']).toarray()

# Subreddit embeddings
subreddit_vocab = author_data['subreddit'].unique()
subreddit_to_idx = {sub: idx for idx, sub in enumerate(subreddit_vocab)}
train_subreddit_idx = train_data['subreddit'].map(subreddit_to_idx).values
val_subreddit_idx = val_data['subreddit'].map(subreddit_to_idx).values

# Create datasets
train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask'],
        'token_type_ids': train_encodings.get('token_type_ids', tf.zeros_like(train_encodings['input_ids'])),
        'tfidf': train_tfidf,
        'subreddit_idx': train_subreddit_idx
    },
    train_data['depression_true'].values
)).shuffle(1000).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': val_encodings['input_ids'],
        'attention_mask': val_encodings['attention_mask'],
        'token_type_ids': val_encodings.get('token_type_ids', tf.zeros_like(val_encodings['input_ids'])),
        'tfidf': val_tfidf,
        'subreddit_idx': val_subreddit_idx
    },
    val_data['depression_true'].values
)).batch(16)

# Define custom model
class DepressionModel(tf.keras.Model):
    def __init__(self, bert_model, num_subreddits, tfidf_dim):
        super(DepressionModel, self).__init__()
        self.bert = bert_model
        self.subreddit_embedding = tf.keras.layers.Embedding(num_subreddits, 16)
        self.tfidf_dense = tf.keras.layers.Dense(64, activation='relu')
        self.concat = tf.keras.layers.Concatenate()
        self.dense1 = tf.keras.layers.Dense(128, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.3)
        self.dense2 = tf.keras.layers.Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):
        bert_output = self.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=inputs['token_type_ids'],
            training=training
        ).pooler_output
        subreddit_emb = self.subreddit_embedding(inputs['subreddit_idx'])
        tfidf_out = self.tfidf_dense(inputs['tfidf'])
        combined = self.concat([bert_output, subreddit_emb, tfidf_out])
        x = self.dense1(combined)
        x = self.dropout(x, training=training)
        return self.dense2(x)

# Initialize model
bert_model = TFBertForSequenceClassification.from_pretrained(model_path, output_attentions=False, output_hidden_states=False)
model = DepressionModel(bert_model.bert, len(subreddit_vocab), 1000)

# Class weights
class_weights = {0: 1.0, 1: 1.0 / prevalence}

# Compile model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# Train model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3,
    class_weight=class_weights
)

# Optimize threshold
val_predictions = model.predict(val_dataset)
val_probs = val_predictions.flatten()
val_true = val_data['depression_true'].values
thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = []
for thresh in thresholds:
    preds = (val_probs >= thresh).astype(int)
    f1 = f1_score(val_true, preds)
    f1_scores.append(f1)
optimal_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold: {optimal_threshold:.2f}")

# Predict on full dataset
full_encodings = tokenize_texts(author_data['text'])
full_tfidf = tfidf.transform(author_data['text']).toarray()
full_subreddit_idx = author_data['subreddit'].map(subreddit_to_idx).values
full_dataset = tf.data.Dataset.from_tensor_slices({
    'input_ids': full_encodings['input_ids'],
    'attention_mask': full_encodings['attention_mask'],
    'token_type_ids': full_encodings.get('token_type_ids', tf.zeros_like(full_encodings['input_ids'])),
    'tfidf': full_tfidf,
    'subreddit_idx': full_subreddit_idx
}).batch(16)

predictions = model.predict(full_dataset).flatten()
pred_df = pd.DataFrame({
    'author': author_data['author'],
    'text': author_data['text'],
    'subreddit': author_data['subreddit'],
    'depression_pred': (predictions >= optimal_threshold).astype(int),
    'depression_prob': predictions
})

# Evaluate
true_labels = author_data['depression_true'].values
pred_labels = pred_df['depression_pred'].values
f1 = f1_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
accuracy = accuracy_score(true_labels, pred_labels)
auc = roc_auc_score(true_labels, pred_df['depression_prob'])
cm = confusion_matrix(true_labels, pred_labels)
tn, fp, fn, tp = cm.ravel()

print("\nDepression Prediction Performance (Per Author):")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Predicted Positive Rate: {pred_df['depression_pred'].mean():.4f}")
print("\nConfusion Matrix:")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")

# Visualizations
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Depressed', 'Depressed'],
            yticklabels=['Not Depressed', 'Depressed'])
plt.title('Confusion Matrix for Depression Prediction (Per Author)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig(os.path.join(output_dir, 'confusion_matrix_improved_author.png'))
plt.close()

fpr, tpr, _ = roc_curve(true_labels, pred_df['depression_prob'])
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve for Depression Prediction (Per Author)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'roc_curve_improved_author.png'))
plt.close()

plt.figure(figsize=(8, 6))
plt.hist(pred_df['depression_prob'], bins=50, alpha=0.7, color='blue', range=(0, 1))
plt.axvline(optimal_threshold, color='red', linestyle='--', label=f'Threshold = {optimal_threshold:.2f}')
plt.title('Distribution of Depression Probabilities (Per Author)')
plt.xlabel('Depression Probability')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'depression_prob_histogram_improved_author.png'))
plt.close()

# Save predictions
pred_df.to_csv(os.path.join(output_dir, 'depression_predictions_improved_author.csv'), index=False)
print(f"\nPredictions saved to: {os.path.join(output_dir, 'depression_predictions_improved_author.csv')}")
print(f"Plots saved to:")
print(f"- {os.path.join(output_dir, 'comment_count_distribution.png')}")
print(f"- {os.path.join(output_dir, 'emotion_prevalence_author.png')}")
print(f"- {os.path.join(output_dir, 'dep_distribution_author.png')}")
print(f"- {os.path.join(output_dir, 'subreddit_distribution_author.png')}")
print(f"- {os.path.join(output_dir, 'confusion_matrix_improved_author.png')}")
print(f"- {os.path.join(output_dir, 'roc_curve_improved_author.png')}")
print(f"- {os.path.join(output_dir, 'depression_prob_histogram_improved_author.png')}")
# ==============================

# ==============================


In [ ]:
#02_length_bias_analysis

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import os
import tensorflow as tf

# ------------------- 1. 计算长度特征 -------------------
print("\n=== 1. 计算文本长度特征 ===")
author_data['total_chars'] = author_data['text'].str.len()
author_data['total_words'] = author_data['text'].str.split().apply(len)
author_data['avg_chars_per_comment'] = author_data['total_chars'] / author_data['comment_count']
author_data['avg_words_per_comment'] = author_data['total_words'] / author_data['comment_count']

# ------------------- 2. 绘制四面板图 -------------------
print("=== 2. 绘制四面板图 ===")
plt.style.use('default')
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('Length Bias Analysis in Depression Prediction', fontsize=16, fontweight='bold')

# (0,0) 总字符分布（log scale）
axes[0,0].hist(author_data['total_chars'], bins=120, color='#1f77b4', alpha=0.8, edgecolor='white')
axes[0,0].set_yscale('log')
axes[0,0].set_xlabel('Total Characters per Author')
axes[0,0].set_ylabel('Frequency (Log Scale)')
axes[0,0].set_title('Distribution of Total Text Length')
max_idx = author_data['total_chars'].idxmax()
axes[0,0].axvline(author_data.loc[max_idx, 'total_chars'], color='red', linestyle='--', linewidth=2)
axes[0,0].text(author_data.loc[max_idx, 'total_chars']*0.92, 1e4,
               f"Max = {author_data.loc[max_idx, 'total_chars']:,}\n({author_data.loc[max_idx, 'comment_count']} comments)",
               color='red', ha='right', fontsize=10)

# (0,1) 箱线图
sns.boxplot(y='total_chars', data=author_data, ax=axes[0,1], color='lightcoral', width=0.4)
axes[0,1].set_ylabel('Total Characters')
axes[0,1].set_title('Boxplot of Total Text Length')

# (1,0) 抑郁 vs 非抑郁
sns.boxplot(x='depression_true', y='total_chars', data=author_data, ax=axes[1,0],
            hue='depression_true', palette={0:'#4c72b0', 1:'#dd8452'}, legend=False)
axes[1,0].set_xticklabels(['Non-Depressed', 'Depressed'])
axes[1,0].set_xlabel('Ground Truth Label')
axes[1,0].set_ylabel('Total Characters')
axes[1,0].set_title('Text Length by Depression Label')

# (1,1) DEDI vs 长度（LOESS）
sns.scatterplot(x='total_chars', y='DEDI', data=author_data, alpha=0.5, ax=axes[1,1], color='gray', s=30)
sns.regplot(x='total_chars', y='DEDI', data=author_data, scatter=False,
            lowess=True, line_kws={'color':'red','linewidth':2}, ax=axes[1,1])
axes[1,1].set_xlabel('Total Characters')
axes[1,1].set_ylabel('DEDI Score')
axes[1,1].set_title('DEDI Score vs Total Text Length (LOESS)')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'length_bias_analysis.png'), dpi=300, bbox_inches='tight')
plt.close()
print("图表已保存 → length_bias_analysis.png")

# ------------------- 3. 统计检验 -------------------
print("\n=== 3. 统计检验 ===")
dep_chars = author_data[author_data['depression_true']==1]['total_chars']
non_chars = author_data[author_data['depression_true']==0]['total_chars']

u_stat, p_val = mannwhitneyu(dep_chars, non_chars, alternative='two-sided')
n1, n2 = len(dep_chars), len(non_chars)
cliffs_delta = (2 * u_stat / (n1 * n2)) - 1 if p_val < 1 else 0

print(f"抑郁组中位数字符数       : {dep_chars.median():.0f}")
print(f"非抑郁组中位数字符数     : {non_chars.median():.0f}")
print(f"Mann-Whitney U p-value   : {p_val:.6f}")
print(f"Cliff's delta            : {cliffs_delta:+.4f}")

# ------------------- 4. 构建长度完全匹配的平衡测试集 -------------------
print("\n=== 4. 构建长度匹配平衡测试集（1:1） ===")
DEDI_df = author_data[author_data['depression_true'] == 1].copy()
non_df = author_data[author_data['depression_true'] == 0].copy()


nn = NearestNeighbors(n_neighbors=1)
nn.fit(DEDI_df[['total_chars']].values)
distances, indices = nn.kneighbors(non_df[['total_chars']].values)

threshold = 8000
mask = distances.flatten() <= threshold
matched_non = non_df.iloc[mask]
matched_DEDI = DEDI_df.iloc[indices.flatten()[mask]]


min_size = min(len(matched_DEDI), len(matched_non))
matched_DEDI = matched_DEDI.sample(n=min_size, random_state=42)
matched_non = matched_non.sample(n=min_size, random_state=42)

matched_test = pd.concat([matched_DEDI, matched_non], ignore_index=True)
matched_test = matched_test.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"成功构建！样本量 = {len(matched_test)} (抑郁 {matched_test['depression_true'].sum()} | 非抑郁 {len(matched_test)//2})")
print(f"长度中位数匹配情况 → 抑郁: {matched_DEDI['total_chars'].median():.0f}  vs  非抑郁: {matched_non['total_chars'].median():.0f}")

# ------------------- 5. 模型预测 -------------------
print("\n=== 5. 在长度匹配测试集上评估原始模型 ===")

# Tokenize
test_enc = tokenize_texts(matched_test['text'])
test_tfidf = tfidf.transform(matched_test['text']).toarray()
test_sub_idx = matched_test['subreddit'].map(subreddit_to_idx)
test_sub_idx = test_sub_idx.fillna(0).astype(int).values

test_dataset = tf.data.Dataset.from_tensor_slices({
    'input_ids': test_enc['input_ids'],
    'attention_mask': test_enc['attention_mask'],
    'token_type_ids': test_enc.get('token_type_ids', tf.zeros_like(test_enc['input_ids'])),
    'tfidf': test_tfidf,
    'subreddit_idx': test_sub_idx
}).batch(16)

probs = model.predict(test_dataset, verbose=0).flatten()
preds = (probs >= optimal_threshold).astype(int)
true_y = matched_test['depression_true'].values

# 完整指标
acc  = accuracy_score(true_y, preds)
prec = precision_score(true_y, preds)
rec  = recall_score(true_y, preds)
f1   = f1_score(true_y, preds)
auc  = roc_auc_score(true_y, probs)
tn, fp, fn, tp = confusion_matrix(true_y, preds).ravel()

print("\n" + "="*70)
print("                     性能对比（Length Bias Ablation）")
print("="*70)
print(f"{'Setting':<25} {'Samples':>8} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>8} {'AUC':>8}")
print("-"*70)
print(f"{'Full Dataset':<25} {len(author_data):8} {accuracy_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)):.4f} "
      f"{precision_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)):.4f} "
      f"{recall_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)):.4f} "
      f"{f1:.4f} {auc:.4f}")
print(f"{'Length-Matched Test':<25} {len(matched_test):8} {acc:.4f} {prec:.4f} {rec:.4f} {f1:.4f} {auc:.4f}")
print("-"*70)
print(f"Confusion Matrix on Length-Matched Test → TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(classification_report(true_y, preds, target_names=['Non-Depressed', 'Depressed']))

# ------------------- 6. 保存结果 -------------------
result_df = pd.DataFrame({
    'Setting': ['Full Dataset', 'Length-Matched Test'],
    'Samples': [len(author_data), len(matched_test)],
    'Accuracy': [accuracy_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)), acc],
    'Precision': [precision_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)), prec],
    'Recall': [recall_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)), rec],
    'F1': [f1_score(author_data['depression_true'], (model.predict(full_dataset).flatten() >= optimal_threshold).astype(int)), f1],
    'AUC': [roc_auc_score(author_data['depression_true'], model.predict(full_dataset).flatten()), auc]
})
result_df.to_csv(os.path.join(output_dir, 'length_bias_ablation_complete.csv'), index=False)


In [ ]:
#03_length_matched_multi_prevalence_eval

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import os
import tensorflow as tf

# ------------------- 1. 计算长度特征 -------------------
print("\n=== 1. 计算文本长度特征（一次） ===")
author_data['total_chars'] = author_data['text'].str.len()
author_data['total_words'] = author_data['text'].str.split().apply(len)
author_data['avg_chars_per_comment'] = author_data['total_chars'] / author_data['comment_count']
author_data['avg_words_per_comment'] = author_data['total_words'] / author_data['comment_count']

# ------------------- 2. 预先构建一次最大的 length-matched 对 -------------------
print("\n=== 2. 构建最大长度匹配对 ===")
DEDI_df_full = author_data[author_data['depression_true'] == 1].copy()
non_df_full = author_data[author_data['depression_true'] == 0].copy()

nn = NearestNeighbors(n_neighbors=1)
nn.fit(DEDI_df_full[['total_chars']].values)
distances, indices = nn.kneighbors(non_df_full[['total_chars']].values)

threshold = 8000
mask = distances.flatten() <= threshold
matched_non_all = non_df_full.iloc[mask].copy()
matched_DEDI_all  = DEDI_df_full.iloc[indices.flatten()[mask]].copy()

print(f"最大匹配对数量 → 抑郁: {len(matched_DEDI_all)}, 非抑郁: {len(matched_non_all)}")

# ------------------- 3. 多 prevalence 循环实验 -------------------
prevalences = [0.05, 0.1102, 0.15]
results_list = []

plt.style.use('default')

for prevalence in prevalences:
    print(f"\n{'='*80}")
    print(f"          Running Prevalence = {prevalence*100:5.2f}%")
    print(f"{'='*80}")
    

    temp_data = author_data.copy()
    temp_data['DEDI'] = (
        temp_data[depression_emotions].sum(axis=1) /
        (temp_data[depression_emotions].sum(axis=1) +
         temp_data[positive_emotions].sum(axis=1) +
         temp_data[neutral_emotions].sum(axis=1) + 1e-8)
    )
    threshold_DEDI = temp_data['DEDI'].quantile(1 - prevalence)
    temp_data['depression_true_prev'] = (temp_data['DEDI'] >= threshold_DEDI).astype(int)
    
    # 把新标签合并回全局 author_data
    author_data['depression_true_prev'] = temp_data['depression_true_prev']
    
    DEDI_curr = matched_DEDI_all[
        matched_DEDI_all['author'].isin(
            author_data[author_data['depression_true_prev'] == 1]['author']
        )
    ].copy()
    
    non_curr = matched_non_all[
        matched_non_all['author'].isin(
            author_data[author_data['depression_true_prev'] == 0]['author']
        )
    ].copy()
    
    min_size = min(len(DEDI_curr), len(non_curr))
    if min_size == 0:
        print(f"Warning: prevalence {prevalence} 下匹配样本不足，跳过！")
        continue
    
    # 采样得到 1:1 且长度匹配的测试集
    matched_DEDI = DEDI_curr.sample(n=min_size, random_state=42)
    matched_non = non_curr.sample(n=min_size, random_state=42)
    
    matched_test = pd.concat([matched_DEDI, matched_non], ignore_index=True)
    matched_test = matched_test.sample(frac=1, random_state=42).reset_index(drop=True)
    

    matched_test = matched_test.merge(
        author_data[['author', 'depression_true_prev']],
        on='author',
        how='left'
    )
    
    print(f"→ Length-matched test set size: {len(matched_test)} "
          f"(Positive: {matched_test['depression_true_prev'].sum()})")
    
    # ------------------- 模型预测 -------------------
    test_enc = tokenize_texts(matched_test['text'])
    test_tfidf = tfidf.transform(matched_test['text']).toarray()
    test_sub_idx = matched_test['subreddit'].map(subreddit_to_idx)
    test_sub_idx = test_sub_idx.fillna(0).astype(int).values
    
    test_dataset = tf.data.Dataset.from_tensor_slices({
        'input_ids': test_enc['input_ids'],
        'attention_mask': test_enc['attention_mask'],
        'token_type_ids': test_enc.get('token_type_ids', tf.zeros_like(test_enc['input_ids'])),
        'tfidf': test_tfidf,
        'subreddit_idx': test_sub_idx
    }).batch(16)
    
    probs = model.predict(test_dataset, verbose=0).flatten()
    preds = (probs >= optimal_threshold).astype(int)
    true_y = matched_test['depression_true_prev'].values
    
    # ------------------- 指标计算 -------------------
    acc  = accuracy_score(true_y, preds)
    prec = precision_score(true_y, preds, zero_division=0)
    rec  = recall_score(true_y, preds, zero_division=0)
    f1   = f1_score(true_y, preds, zero_division=0)
    auc  = roc_auc_score(true_y, probs)
    tn, fp, fn, tp = confusion_matrix(true_y, preds).ravel()
    
    results_list.append({
        'Prevalence': f"{prevalence*100:.2f}%",
        'Samples': len(matched_test),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'AUC': auc,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    })
    
    print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    
    # ------------------- 绘图 -------------------
    prev_str = f"prev_{prevalence*100:.2f}"
    
    # 1. Confusion Matrix
    plt.figure(figsize=(6,5))
    sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Depressed', 'Depressed'],
                yticklabels=['Non-Depressed', 'Depressed'], cbar=False)
    plt.title(f'Confusion Matrix\n(Prevalence = {prevalence*100:.2f}%, Length-Matched)')
    plt.xlabel('Predicted')
    plt.ylabel('Ground Truth')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'confusion_matrix_{prev_str}_lengthmatched.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(true_y, probs)
    plt.figure(figsize=(7,6))
    plt.plot(fpr, tpr, label=f'ROC (AUC = {auc:.4f})', color='#1f77b4', linewidth=2.5)
    plt.plot([0,1],[0,1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve\n(Prevalence = {prevalence*100:.2f}%, Length-Matched)')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'roc_curve_{prev_str}_lengthmatched.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Probability Distribution
    plt.figure(figsize=(8,5))
    plt.hist(probs[true_y==0], bins=40, alpha=0.7, label='Non-Depressed', color='#4c72b0')
    plt.hist(probs[true_y==1], bins=40, alpha=0.7, label='Depressed', color='#dd8452')
    plt.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {optimal_threshold:.2f}')
    plt.xlabel('Predicted Depression Probability')
    plt.ylabel('Frequency')
    plt.title(f'Prediction Probability Distribution\n(Prevalence = {prevalence*100:.2f}%, Length-Matched)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'prob_distribution_{prev_str}_lengthmatched.png'), dpi=300, bbox_inches='tight')
    plt.close()

# ------------------- 4. 汇总结果 -------------------
result_df = pd.DataFrame(results_list)
result_df.to_csv(os.path.join(output_dir, 'length_matched_performance_across_prevalence.csv'), index=False)

print("\n" + "="*80)

for p in prevalences:
    ps = f"prev_{p*100:.2f}"
    print(f"\nPrevalence {p*100:.2f}%:")
    print(f"   • confusion_matrix_{ps}_lengthmatched.png")
    print(f"   • roc_curve_{ps}_lengthmatched.png")
    print(f"   • prob_distribution_{ps}_lengthmatched.png")
print("\n汇总表格已保存：length_matched_performance_across_prevalence.csv")
print("路径：", output_dir)
print("="*80)


In [ ]:
#04_publication_figures

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, auc
import os


plt.style.use('default')
sns.set_style("whitegrid")
sns.set_palette("colorblind")  
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11


summary_df = pd.read_csv(os.path.join(output_dir, 'length_matched_performance_across_prevalence.csv'))

# 预定义颜色
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # blue, orange, green
prev_values = [0.05, 0.1102, 0.15]
color_map = dict(zip([f"{p*100:.2f}%" for p in prev_values], colors))

# ------------------- 1. DEDIp_distribution_prev  -------------------
print("\n正在绘制 → fig_DEDI_distribution_all_lengthmatched.png")
plt.figure(figsize=(8, 5.5))
for prevalence in prevalences:
    prev_str = f"{prevalence*100:.2f}"
    # 重新计算当前 prevalence 下的 DEDI 和标签
    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (
        temp[depression_emotions].sum(axis=1) + 
        temp[positive_emotions].sum(axis=1) + 
        temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['label'] = (temp['DEDI'] >= threshold).astype(int)
    

    curr_authors = set(matched_DEDI_all['author']) | set(matched_non_all['author'])
    mask = temp['author'].isin(curr_authors)
    DEDI_scores = temp.loc[mask, 'DEDI']
    
    sns.kdeplot(DEDI_scores, fill=True, alpha=0.55, label=f'{prevalence*100:.2f}%', color=color_map[f"{prevalence*100:.2f}%"], linewidth=2)

plt.xlim(0, 1)
plt.xlabel('DEDI Score')
plt.ylabel('Density')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_DEDI_distribution_all_lengthmatched.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 2. confusion_matrix_prev  -------------------
print("正在绘制 → fig_confusion_matrix_all_lengthmatched.png")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, prevalence in zip(axes, prevalences):
    prev_str = f"{prevalence*100:.2f}"
    path = os.path.join(output_dir, f'confusion_matrix_prev_{prev_str}_lengthmatched.png')

    row = summary_df[summary_df['Prevalence'] == f"{prevalence*100:.2f}%"].iloc[0]
    cm = np.array([[row['TN'], row['FP']], [row['FN'], row['TP']]])
    total = cm.sum()
    cm_percent = cm / total * 100
    
    sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Non-Depressed', 'Depressed'],
                yticklabels=['Non-Depressed', 'Depressed'], annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title(f'{prevalence*100:.2f}%', pad=15, size=13)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_confusion_matrix_all_lengthmatched.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 3. ROC Curve 三条合一 -------------------
print("正在绘制 → fig_roc_curve_all_lengthmatched.png")
plt.figure(figsize=(7.5, 6.5))
for prevalence in prevalences:

    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (
        temp[depression_emotions].sum(axis=1) + temp[positive_emotions].sum(axis=1) + 
        temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['depression_true_prev'] = (temp['DEDI'] >= threshold).astype(int)
    
    # length-matched 作者集合
    curr_authors = set(matched_DEDI_all['author']) | set(matched_non_all['author'])
    sub = temp[temp['author'].isin(curr_authors)]
    # 预测
    enc = tokenize_texts(sub['text'])
    tfidf_feat = tfidf.transform(sub['text']).toarray()
    sub_idx = sub['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices({
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])),
        'tfidf': tfidf_feat,
        'subreddit_idx': sub_idx
    }).batch(16)
    probs = model.predict(ds, verbose=0).flatten()
    true_y = sub['depression_true_prev'].values
    
    fpr, tpr, _ = roc_curve(true_y, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{prevalence*100:.2f}% (AUC = {roc_auc:.3f})', 
             color=color_map[f"{prevalence*100:.2f}%"], linewidth=2.5)

plt.plot([0,1],[0,1], 'k--', linewidth=1)
plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_roc_curve_all_lengthmatched.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 4. PR Curve 三条合一 -------------------
print("正在绘制 → fig_pr_curve_all_lengthmatched.png")
plt.figure(figsize=(7.5, 6.5))
for prevalence in prevalences:
    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (
        temp[depression_emotions].sum(axis=1) + temp[positive_emotions].sum(axis=1) + 
        temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['depression_true_prev'] = (temp['DEDI'] >= threshold).astype(int)
    curr_authors = set(matched_DEDI_all['author']) | set(matched_non_all['author'])
    sub = temp[temp['author'].isin(curr_authors)]
    
    enc = tokenize_texts(sub['text'])
    tfidf_feat = tfidf.transform(sub['text']).toarray()
    sub_idx = sub['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices({
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])),
        'tfidf': tfidf_feat,
        'subreddit_idx': sub_idx
    }).batch(16)
    probs = model.predict(ds, verbose=0).flatten()
    true_y = sub['depression_true_prev'].values
    
    precision, recall, _ = precision_recall_curve(true_y, probs)
    pr_auc = auc(recall, precision)
    plt.plot(recall, precision, label=f'{prevalence*100:.2f}% (AUPRC = {pr_auc:.3f})', 
             color=color_map[f"{prevalence*100:.2f}%"], linewidth=2.5)

plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_pr_curve_all_lengthmatched.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 5. Prevalence Sensitivity (F1/Precision/Recall vs Prevalence) -------------------
print("正在绘制 → fig_prevalence_sensitivity_f1.png")
plt.figure(figsize=(7.5, 5.5))
plt.plot(summary_df['Prevalence'], summary_df['F1'], 'o-', color='#d62728', linewidth=2.5, markersize=8, label='F1 Score')
plt.plot(summary_df['Prevalence'], summary_df['Precision'], 's--', color='#1f77b4', linewidth=2.5, markersize=8, label='Precision')
plt.plot(summary_df['Prevalence'], summary_df['Recall'], '^:', color='#ff7f0e', linewidth=2.5, markersize=9, label='Recall')
plt.xlabel('Prevalence')
plt.ylabel('Score')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_prevalence_sensitivity_f1.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 6. AUC & AUPRC vs Prevalence -------------------
print("正在绘制 → fig_prevalence_sensitivity_auc.png")
plt.figure(figsize=(7.5, 5.5))
plt.plot(summary_df['Prevalence'], summary_df['AUC'], 'o-', color='#2ca02c', linewidth=3, markersize=9, label='AUROC')

auprc_list = []
for prevalence in prevalences:
    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (temp[depression_emotions].sum(axis=1) + temp[positive_emotions].sum(axis=1) + temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['label'] = (temp['DEDI'] >= threshold).astype(int)
    sub = temp[temp['author'].isin(set(matched_DEDI_all['author']) | set(matched_non_all['author']))]
    enc = tokenize_texts(sub['text'])
    tfidf_f = tfidf.transform(sub['text']).toarray()
    sub_i = sub['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices({'input_ids': enc['input_ids'],'attention_mask': enc['attention_mask'],
        'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])), 'tfidf': tfidf_f, 'subreddit_idx': sub_i}).batch(16)
    p = model.predict(ds, verbose=0).flatten()
    prec, rec, _ = precision_recall_curve(sub['label'], p)
    auprc_list.append(auc(rec, prec))
plt.plot(summary_df['Prevalence'], auprc_list, 's--', color='#9467bd', linewidth=3, markersize=9, label='AUPRC')
plt.xlabel('Prevalence')
plt.ylabel('Area Under Curve')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_prevalence_sensitivity_auc.png'), dpi=300, bbox_inches='tight')
plt.close()



from sklearn.calibration import calibration_curve
print("正在绘制 → fig_calibration_curve_lengthmatched.png")
plt.figure(figsize=(7, 6))
for prevalence in prevalences:
    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (temp[depression_emotions].sum(axis=1) + temp[positive_emotions].sum(axis=1) + temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['label'] = (temp['DEDI'] >= threshold).astype(int)
    sub = temp[temp['author'].isin(set(matched_DEDI_all['author']) | set(matched_non_all['author']))]
    enc = tokenize_texts(sub['text'])
    tfidf_f = tfidf.transform(sub['text']).toarray()
    sub_i = sub['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices({'input_ids': enc['input_ids'],'attention_mask': enc['attention_mask'],
        'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])), 'tfidf': tfidf_f, 'subreddit_idx': sub_i}).batch(16)
    probs = model.predict(ds, verbose=0).flatten()
    fraction_of_positives, mean_predicted_value = calibration_curve(sub['label'], probs, n_bins=10)
    plt.plot(mean_predicted_value, fraction_of_positives, "s-", label=f'{prevalence*100:.2f}%', color=color_map[f"{prevalence*100:.2f}%"])
plt.plot([0,1],[0,1], "k:", label="Perfectly calibrated")
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_calibration_curve_lengthmatched.png'), dpi=300, bbox_inches='tight')
plt.close()

# 7.2 Decision Threshold Sensitivity（不同阈值下的 F1）
print("正在绘制 → fig_threshold_sensitivity_lengthmatched.png")
plt.figure(figsize=(8, 5.5))
thresholds = np.linspace(0.05, 0.95, 200)
for prevalence in prevalences:
    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (temp[depression_emotions].sum(axis=1) + temp[positive_emotions].sum(axis=1) + temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['label'] = (temp['DEDI'] >= threshold).astype(int)
    sub = temp[temp['author'].isin(set(matched_DEDI_all['author']) | set(matched_non_all['author']))]
    enc = tokenize_texts(sub['text'])
    tfidf_f = tfidf.transform(sub['text']).toarray()
    sub_i = sub['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices({'input_ids': enc['input_ids'],'attention_mask': enc['attention_mask'],
        'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])), 'tfidf': tfidf_f, 'subreddit_idx': sub_i}).batch(16)
    probs = model.predict(ds, verbose=0).flatten()
    true_y = sub['label'].values
    f1s = [f1_score(true_y, (probs >= t).astype(int)) for t in thresholds]
    plt.plot(thresholds, f1s, label=f'{prevalence*100:.2f}%', color=color_map[f"{prevalence*100:.2f}%"], linewidth=2.5)
plt.axvline(optimal_threshold, color='red', linestyle='--', linewidth=1.5, label=f'Optimal = {optimal_threshold:.2f}')
plt.xlabel('Decision Threshold')
plt.ylabel('F1 Score')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_threshold_sensitivity_lengthmatched.png'), dpi=300, bbox_inches='tight')
plt.close()


print("   • fig_DEDI_distribution_all_lengthmatched.png")
print("   • fig_confusion_matrix_all_lengthmatched.png")
print("   • fig_roc_curve_all_lengthmatched.png")
print("   • fig_pr_curve_all_lengthmatched.png")
print("   • fig_prevalence_sensitivity_f1.png")
print("   • fig_prevalence_sensitivity_auc.png")
print("   • fig_calibration_curve_lengthmatched.png")
print("   • fig_threshold_sensitivity_lengthmatched.png")
print("全部保存在：", output_dir)




# ------------------- 2. Confusion Matrix--------------------

raw_data = {
    "5.00%":  {"TN": 2156, "TP": 2078, "FP":  90, "FN":  166},
    "11.02%": {"TN": 4328, "FP":  219, "FN": 419, "TP": 4128},
    "15.00%": {"TN": 4175, "FP":  177, "FN": 400, "TP": 3952}
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
fig.subplots_adjust(wspace=0.4)

for ax, (prev_str, vals) in zip(axes, raw_data.items()):
    # 2×2 混淆矩阵
    cm = np.array([[vals["TN"], vals["FP"]],
                   [vals["FN"], vals["TP"]]])
    

    cm_percent = cm / cm.sum(axis=1, keepdims=True) * 100
    

    labels = np.char.add(
        np.round(cm_percent, 1).astype(str),
        np.full((2,2), "%")
    )
    
    sns.heatmap(cm_percent, annot=labels, fmt='', cmap="Blues", cbar=False, ax=ax,
                xticklabels=['Non-Depressed', 'Depressed'],
                yticklabels=['Non-Depressed', 'Depressed'],
                annot_kws={"size": 18, "weight": "bold", "color": "black"},
                linewidths=2, linecolor='white',
                alpha=0.95)
    
    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('Ground Truth', fontsize=14)
    ax.set_title(f'{prev_str}', pad=25, fontsize=15, fontweight='bold')


plt.suptitle('', fontsize=0)
plt.tight_layout()


save_path = os.path.join(output_dir, 'fig_confusion_matrix_percent_only_lengthmatched.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()

print("已生成混淆矩阵图：")
print(save_path)

In [ ]:
#05_error_analysis

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import tensorflow as tf
import os

# ------------------- 设置美观风格 -------------------
plt.style.use('default')
sns.set_style("whitegrid")
sns.set_palette("colorblind")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'Arial'


prevalence = 0.1102

# 重新计算 DEDI 和真实标签
temp_data = author_data.copy()
temp_data['DEDI'] = (
    temp_data[depression_emotions].sum(axis=1) /
    (temp_data[depression_emotions].sum(axis=1) +
     temp_data[positive_emotions].sum(axis=1) +
     temp_data[neutral_emotions].sum(axis=1) + 1e-8)
)
threshold_DEDIp = temp_data['DEDI'].quantile(1 - prevalence)
temp_data['depression_true_1102'] = (temp_data['DEDI'] >= threshold_DEDI).astype(int)


matched_authors = set(matched_DEDI_all['author']) | set(matched_non_all['author'])
matched_test = temp_data[temp_data['author'].isin(matched_authors)].copy()

# 模型预测
enc = tokenize_texts(matched_test['text'])
tfidf_feat = tfidf.transform(matched_test['text']).toarray()
sub_idx = matched_test['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values

ds = tf.data.Dataset.from_tensor_slices({
    'input_ids': enc['input_ids'],
    'attention_mask': enc['attention_mask'],
    'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])),
    'tfidf': tfidf_feat,
    'subreddit_idx': sub_idx
}).batch(16)

probs_1102 = model.predict(ds, verbose=0).flatten()
preds_1102 = (probs_1102 >= optimal_threshold).astype(int)
true_1102 = matched_test['depression_true_1102'].values

# 标记错误类型
matched_test['pred'] = preds_1102
matched_test['prob'] = probs_1102
matched_test['error_type'] = 'Correct'
matched_test.loc[(true_1102 == 1) & (preds_1102 == 0), 'error_type'] = 'FN'
matched_test.loc[(true_1102 == 0) & (preds_1102 == 1), 'error_type'] = 'FP'

fp_texts = matched_test[matched_test['error_type'] == 'FP']['text']
fn_texts = matched_test[matched_test['error_type'] == 'FN']['text']

# ------------------- 图1: FP vs FN 词云对比 -------------------
print("正在生成 → fig_error_analysis_wordcloud.png")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))


def preprocess(text):
    return re.findall(r'\b[a-zA-Z]+\b', text.lower())

fp_words = ' '.join([' '.join(preprocess(t)) for t in fp_texts])
fn_words = ' '.join([' '.join(preprocess(t)) for t in fn_texts])

# 词云
wc_fp = WordCloud(width=800, height=600, background_color='white', 
                  colormap='Reds', max_words=150, min_font_size=10, 
                  random_state=42).generate(fp_words)
ax1.imshow(wc_fp, interpolation='bilinear')
ax1.axis('off')
ax1.text(0.5, 0.02, 'False Positives (FP)', transform=ax1.transAxes, 
         ha='center', va='bottom', fontsize=16, fontweight='bold')

wc_fn = WordCloud(width=800, height=600, background_color='white', 
                  colormap='Blues', max_words=150, min_font_size=10, 
                  random_state=42).generate(fn_words)
ax2.imshow(wc_fn, interpolation='bilinear')
ax2.axis('off')
ax2.text(0.5, 0.02, 'False Negatives (FN)', transform=ax2.transAxes, 
         ha='center', va='bottom', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_error_analysis_wordcloud.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 图2: 错误样本的预测概率分布 -------------------
print("正在生成 → fig_error_prob_distribution.png")
plt.figure(figsize=(8, 5.5))

sns.histplot(data=matched_test[matched_test['error_type']=='FP'], x='prob', 
             color='#d62728', alpha=0.7, bins=30, label='False Positives', stat='density')
sns.histplot(data=matched_test[matched_test['error_type']=='FN'], x='prob', 
             color='#1f77b4', alpha=0.7, bins=30, label='False Negatives', stat='density')
sns.histplot(data=matched_test[matched_test['error_type']=='Correct'], x='prob', 
             color='gray', alpha=0.3, bins=50, label='Correct', stat='density')

plt.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {optimal_threshold:.2f}')
plt.xlabel('Predicted Depression Probability')
plt.ylabel('Density')
plt.legend(frameon=True, fancybox=False, edgecolor='black')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_error_prob_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()


# ------------------- 图3: 稀有情绪在 FP/FN 中的富集热图 -------------------
print("正在生成 → fig_rare_emotion_heatmap.png")
# 稀有情绪定义
rare_emotions = ['grief', 'pride', 'relief', 'nervousness']

# 计算每类样本中每种稀有情绪出现的比例
def enrichment_ratio(group):
    total = len(group)
    ratios = {}
    for emo in rare_emotions:
        count = group[emo].sum()
        overall_rate = author_data[emo].mean()
        ratio = (count / total) / overall_rate if overall_rate > 0 else 0
        ratios[emo] = ratio
    return ratios

fp_ratios = enrichment_ratio(matched_test[matched_test['error_type']=='FP'])
fn_ratios = enrichment_ratio(matched_test[matched_test['error_type']=='FN'])
correct_ratios = enrichment_ratio(matched_test[matched_test['error_type']=='Correct'])

df_enrich = pd.DataFrame([correct_ratios, fp_ratios, fn_ratios], 
                         index=['Correct', 'False Positive', 'False Negative'])

plt.figure(figsize=(8, 4))
sns.heatmap(df_enrich, annot=True, fmt='.2f', cmap='RdBu_r', center=1.0, 
            cbar_kws={'label': 'Enrichment Ratio'}, linewidths=0.5)
plt.ylabel('')
plt.xlabel('Rare Emotions')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_rare_emotion_heatmap.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 图4: t-SNE 可视化 -------------------
# print("正在生成 → fig_tsne_projection.png")
# # 使用 BERT 的 [CLS] 表征作为特征
# bert_model_eval = model.bert
# cls_features = []

# for batch in ds:
#     bert_out = bert_model_eval(batch, training=False).pooler_output
#     cls_features.append(bert_out.numpy())

# cls_features = np.vstack(cls_features)

# # t-SNE
# tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
# tsne_proj = tsne.fit_transform(cls_features)

# matched_test['tsne_x'] = tsne_proj[:, 0]
# matched_test['tsne_y'] = tsne_proj[:, 1]

# plt.figure(figsize=(9, 8))
# palette = {'Correct': 'lightgray', 'FP': '#d62728', 'FN': '#1f77b4'}

# # 正确样本（淡化）
# plt.scatter(matched_test[matched_test['error_type']=='Correct']['tsne_x'],
#             matched_test[matched_test['error_type']=='Correct']['tsne_y'],
#             c='lightgray', alpha=0.6, s=30, label='Correct')

# # 错误样本（高亮）
# for err_type, color in [('FP', '#d62728'), ('FN', '#1f77b4')]:
#     subset = matched_test[matched_test['error_type'] == err_type]
#     plt.scatter(subset['tsne_x'], subset['tsne_y'], 
#                 c=color, alpha=0.9, s=60, edgecolors='black', linewidth=0.5, label=err_type)

# plt.xlabel('t-SNE Dimension 1')
# plt.ylabel('t-SNE Dimension 2')
# plt.legend(frameon=True, fancybox=False, edgecolor='black', markerscale=1.5)
# sns.despine()
# plt.tight_layout()
# plt.savefig(os.path.join(output_dir, 'fig_tsne_projection.png'), dpi=300, bbox_inches='tight')
# plt.close()



print("\n错误分析 4 张图全部生成完毕（prevalence=11.02%，Length-Matched）：")
print("   • fig_error_analysis_wordcloud.png")
print("   • fig_error_prob_distribution.png")
print("   • fig_rare_emotion_heatmap.png")
#print("   • fig_tsne_projection.png")
print("路径：", output_dir)

In [ ]:
#06_final_visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

# 设置美观风格
plt.style.use('default')
sns.set_style("white")
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

# 配色
colors = {
    '5.00%': '#E74C3C',   # 红
    '11.02%': '#27AE60',  # 绿（主实验色）
    '15.00%': '#3498DB'   # 蓝
}

# ------------------- 图1: 三种 prevalence 的 DEDI 分布 + 阈值标注 -------------------
print("正在生成 → fig_DEDI_distribution_combined.png")

plt.figure(figsize=(9, 6))

prevalences = [0.05, 0.1102, 0.15]
thresholds = []
labels = []

for i, prev in enumerate(prevalences):
    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (
        temp[depression_emotions].sum(axis=1) + 
        temp[positive_emotions].sum(axis=1) + 
        temp[neutral_emotions].sum(axis=1) + 1e-8
    )
    threshold = temp['DEDI'].quantile(1 - prev)
    thresholds.append(threshold)
    
    # 使用 length-matched 样本池中的 DEDI 分布
    matched_authors = set(matched_DEDI_all['author']) | set(matched_non_all['author'])
    DEDI_scores = temp[temp['author'].isin(matched_authors)]['DEDI']
    
    sns.kdeplot(DEDI_scores, fill=True, alpha=0.65, linewidth=2.5,
                color=colors[f"{prev*100:.2f}%"], label=f'{prev*100:.2f}%')

# 绘制三条阈值虚线
y_positions = [0.92, 0.85, 0.78]  
for thresh, y_pos, prev in zip(thresholds, y_positions, prevalences):
    plt.axvline(thresh, color=colors[f"{prev*100:.2f}%"], linestyle='--', linewidth=2.2, alpha=0.9)
    plt.text(thresh + 0.008, plt.ylim()[1] * y_pos,
             f'{thresh:.3f}', 
             color=colors[f"{prev*100:.2f}%"], fontsize=13, fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9, edgecolor='none'))

plt.xlim(0, 1)
plt.xlabel('DEDI Score')
plt.ylabel('Density')
plt.legend(title='Prevalence', frameon=True, fancybox=False, edgecolor='black')
sns.despine(trim=True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_DEDI_distribution_combined.png'), dpi=300, bbox_inches='tight')
plt.close()

# ------------------- 图2:  prevalence=11.02% 的预测概率分布 -------------------
print("正在生成 → fig_prob_distribution_1102.png")

prevalence = 0.1102
temp = author_data.copy()
temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (
    temp[depression_emotions].sum(axis=1) + 
    temp[positive_emotions].sum(axis=1) + 
    temp[neutral_emotions].sum(axis=1) + 1e-8
)
threshold_DEDI = temp['DEDI'].quantile(1 - prevalence)
temp['depression_true'] = (temp['DEDI'] >= threshold_DEDI).astype(int)

matched_authors = set(matched_DEDI_all['author']) | set(matched_non_all['author'])
sub = temp[temp['author'].isin(matched_authors)].copy()

# 模型预测
enc = tokenize_texts(sub['text'])
tfidf_f = tfidf.transform(sub['text']).toarray()
sub_idx = sub['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
ds = tf.data.Dataset.from_tensor_slices({
    'input_ids': enc['input_ids'],
    'attention_mask': enc['attention_mask'],
    'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])),
    'tfidf': tfidf_f,
    'subreddit_idx': sub_idx
}).batch(16)

probs = model.predict(ds, verbose=0).flatten()
true_labels = sub['depression_true'].values

plt.figure(figsize=(9, 6))

# 非抑郁 vs 抑郁 双峰分布
sns.histplot(probs[true_labels == 0], bins=50, alpha=0.75, color='#3498DB', 
             label='Non-Depressed', kde=True, linewidth=0, stat='density')
sns.histplot(probs[true_labels == 1], bins=50, alpha=0.75, color='#E74C3C', 
             label='Depressed', kde=True, linewidth=0, stat='density')

# 最优阈值线 + 标注
plt.axvline(optimal_threshold, color='black', linestyle='--', linewidth=2.5)
plt.text(optimal_threshold + 0.01, plt.ylim()[1] * 0.92,
         f'Threshold = {optimal_threshold:.3f}', 
         color='black', fontsize=13, fontweight='bold',
         bbox=dict(boxstyle="round,pad=0.4", facecolor='white', edgecolor='black', alpha=0.95))

plt.xlabel('Predicted Depression Probability')
plt.ylabel('Density')
plt.legend(title='Ground Truth (Length-Matched)', frameon=True, fancybox=False, edgecolor='black')
sns.despine(trim=True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_prob_distribution_1102.png'), dpi=300, bbox_inches='tight')
plt.close()

print("\n两张图已生成：")
print("   → fig_DEDI_distribution_combined.png    (三阈值)")
print("   → fig_prob_distribution_1102.png       (11.02% 主实验概率分布)")
print("路径：", output_dir)







from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8.5, 7))
colors = ['#E74C3C', '#27AE60', '#3498DB']
prevalences = [0.05, 0.1102, 0.15]
labels = ['5.00%', '11.02%', '15.00%']


summary_df = pd.read_csv(os.path.join(output_dir, 'length_matched_performance_across_prevalence.csv'))
auc_dict = dict(zip(summary_df['Prevalence'], summary_df['AUC']))

for i, prevalence in enumerate(prevalences):
    print(f"正在绘制 prevalence = {prevalence*100:.2f}% 的 ROC（使用严格 length-matched 数据）")
    

    temp = author_data.copy()
    temp['DEDI'] = temp[depression_emotions].sum(axis=1) / (
        temp[depression_emotions].sum(axis=1) + 
        temp[positive_emotions].sum(axis=1) + 
        temp[neutral_emotions].sum(axis=1) + 1e-8)
    threshold = temp['DEDI'].quantile(1 - prevalence)
    temp['depression_true_prev'] = (temp['DEDI'] >= threshold).astype(int)
    

    DEDI_curr = matched_DEDI_all[matched_DEDI_all['author'].isin(temp[temp['depression_true_prev']==1]['author'])]
    non_curr = matched_non_all[matched_non_all['author'].isin(temp[temp['depression_true_prev']==0]['author'])]
    min_size = min(len(DEDI_curr), len(non_curr))
    matched_DEDI = DEDI_curr.sample(n=min_size, random_state=42)
    matched_non = non_curr.sample(n=min_size, random_state=42)
    matched_test = pd.concat([matched_DEDI, matched_non]).sample(frac=1, random_state=42)
    
    # 合并真实标签
    matched_test = matched_test.merge(temp[['author', 'depression_true_prev']], on='author', how='left')
    
    # 预测
    enc = tokenize_texts(matched_test['text'])
    tfidf_f = tfidf.transform(matched_test['text']).toarray()
    sub_idx = matched_test['subreddit'].map(subreddit_to_idx).fillna(0).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices({
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'token_type_ids': enc.get('token_type_ids', tf.zeros_like(enc['input_ids'])),
        'tfidf': tfidf_f,
        'subreddit_idx': sub_idx
    }).batch(16)
    
    probs = model.predict(ds, verbose=0).flatten()
    true_y = matched_test['depression_true_prev'].values
    
    fpr, tpr, _ = roc_curve(true_y, probs)
    roc_auc = auc(fpr, tpr)
    
    # 直接使用表格里的 AUC
    table_auc = auc_dict[f"{prevalence*100:.2f}%"]
    plt.plot(fpr, tpr, color=colors[i], linewidth=3,
             label=f'{labels[i]} (AUC = {table_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.8)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right", frameon=True, fancybox=False, edgecolor='black')
sns.despine(trim=True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_roc_curve_all_lengthmatched_corrected.png'), dpi=300, bbox_inches='tight')
plt.close()

print("ROC 已生成：fig_roc_curve_all_lengthmatched_corrected.png")
